# RQ1 — Performance Evaluation (Dataset A)

**Nội dung:**
- Threshold-independent metrics (ROC-AUC, PR-AUC, Brier)
- Calibration assessment (curve, intercept, slope)
- Threshold-dependent metrics (BA, Precision/Recall/F1 per class)
- Paired Bootstrap CI (1000 resamples, 95% CI)
- Table 11.1 + Calibration curves

> ⚠️ **Chạy trước:** `LogisticRegression.ipynb` và `RandomForest.ipynb`

In [ ]:
import sys, time, warnings, joblib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.path import DATA_DIR

DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "datasetA"
RESULTS_DIR = DATASET_DIR / "results"
TABLES_DIR  = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR  = RESULTS_DIR / "models"
for d in (TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL   = "action_taken"
FEATURE_COLS = ["applicant_sex","income","loan_amount","loan_purpose","lien_status","loan_type","derived_race"]
NUM_COLS     = ["income", "loan_amount"]
GROUP_NAMES  = ["applicant_race","applicant_sex","applicant_income","loan_amount","loan_purpose","lien_status","loan_type"]

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")

## 0. Load Models & Data

In [ ]:
# Load trained models + metadata
lr_model = joblib.load(MODELS_DIR / "lr_model.pkl")
rf_model = joblib.load(MODELS_DIR / "rf_model.pkl")
scaler   = joblib.load(MODELS_DIR / "scaler.pkl")
lr_meta  = joblib.load(MODELS_DIR / "lr_meta.pkl")
rf_meta  = joblib.load(MODELS_DIR / "rf_meta.pkl")

lr_threshold = lr_meta["threshold"]
rf_threshold = rf_meta["threshold"]
lr_train_time = lr_meta["train_time"]
rf_train_time = rf_meta["train_time"]
lr_infer_time = lr_meta["infer_time"]
rf_infer_time = rf_meta["infer_time"]

print(f"LR threshold: {lr_threshold}")
print(f"RF threshold: {rf_threshold}")

# Load test data + scale
def load_split(name):
    df = pd.read_csv(DATASET_DIR / name / f"{name}.csv")
    return df[FEATURE_COLS].copy(), df[TARGET_COL].copy()

X_test, y_test_s = load_split("test")
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])
y_test = y_test_s.values

# Probabilities on test set
lr_prob_test = lr_model.predict_proba(X_test)[:, 1]
rf_prob_test = rf_model.predict_proba(X_test)[:, 1]
print(f"Test set size: {len(y_test):,}")

---
# PART A — RQ1: Paired Bootstrap CI (Mục 6)

1000 bootstrap resamples trên Test set, cùng bộ chỉ số cho LR và RF.
Threshold giữ cố định, không refit model.

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression as _LR_Cal

def calibration_intercept_slope(y_true, y_prob):
    p = np.clip(y_prob, 1e-10, 1 - 1e-10)
    logit_p = np.log(p / (1 - p)).reshape(-1, 1)
    cal = _LR_Cal(penalty=None, solver="lbfgs", max_iter=1000)
    cal.fit(logit_p, y_true)
    return float(cal.intercept_[0]), float(cal.coef_[0, 0])

def compute_rq1_metrics(y, p, t):
    yhat = (p >= t).astype(int)
    y_den = 1 - y; p_den = 1 - p
    cal_int, cal_slope = calibration_intercept_slope(y, p)
    return {
        "ROC-AUC":             roc_auc_score(y, p),
        "PR-AUC (Denied)":     average_precision_score(y_den, p_den),
        "Brier Score":         brier_score_loss(y, p),
        "Cal Intercept":       cal_int,
        "Cal Slope":           cal_slope,
        "Balanced Accuracy":   balanced_accuracy_score(y, yhat),
        "Precision (Approved)":precision_score(y, yhat, pos_label=1, zero_division=0),
        "Recall (Approved)":   recall_score(y, yhat, pos_label=1, zero_division=0),
        "F1 (Approved)":       f1_score(y, yhat, pos_label=1, zero_division=0),
        "Precision (Denied)":  precision_score(y, yhat, pos_label=0, zero_division=0),
        "Recall (Denied)":     recall_score(y, yhat, pos_label=0, zero_division=0),
        "F1 (Denied)":         f1_score(y, yhat, pos_label=0, zero_division=0),
    }

# Point estimates
lr_metrics = compute_rq1_metrics(y_test, lr_prob_test, lr_threshold)
rf_metrics = compute_rq1_metrics(y_test, rf_prob_test, rf_threshold)
metric_keys = list(lr_metrics.keys())

print("Point estimates computed ✓")

In [ ]:
# ── Paired Bootstrap (Mục 6) ──
N_BOOTSTRAP = 1_000
rng = np.random.default_rng(RANDOM_STATE)
boot_idx = rng.integers(0, len(y_test), size=(N_BOOTSTRAP, len(y_test)))

lr_boot = {k: [] for k in metric_keys}
rf_boot = {k: [] for k in metric_keys}

print(f"Running {N_BOOTSTRAP:,} Paired Bootstrap iterations for RQ1...")
t0 = time.perf_counter()

for b in range(N_BOOTSTRAP):
    idx = boot_idx[b]
    yb = y_test[idx]
    lr_m = compute_rq1_metrics(yb, lr_prob_test[idx], lr_threshold)
    rf_m = compute_rq1_metrics(yb, rf_prob_test[idx], rf_threshold)
    for k in metric_keys:
        lr_boot[k].append(lr_m[k])
        rf_boot[k].append(rf_m[k])

elapsed_rq1_boot = time.perf_counter() - t0

def ci(arr):
    return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))

lr_ci = {k: ci(lr_boot[k]) for k in metric_keys}
rf_ci = {k: ci(rf_boot[k]) for k in metric_keys}
print(f"✓ Paired bootstrap completed in {elapsed_rq1_boot:.2f}s")

### Table 11.1 — RQ1 Performance (Mục 11.1)

In [ ]:
# ── Table 11.1 ──
rows = []
for m_name, met, ci_d, thr, tt, it in [
    ("Logistic Regression", lr_metrics, lr_ci, lr_threshold, lr_train_time, lr_infer_time),
    ("Random Forest",       rf_metrics, rf_ci, rf_threshold, rf_train_time, rf_infer_time),
]:
    row = {"Model": m_name, "Threshold": thr}
    for k in metric_keys:
        row[k] = f"{met[k]:.4f}"
        row[f"{k} 95% CI"] = f"[{ci_d[k][0]:.4f}, {ci_d[k][1]:.4f}]"
    row["Training Time (s)"]  = f"{tt:.1f}"
    row["Inference Time (s)"] = f"{it:.4f}"
    rows.append(row)

df_rq1 = pd.DataFrame(rows)
df_rq1.to_csv(TABLES_DIR / "table_11_1_rq1_performance.csv", index=False)

# Display
for _, r in df_rq1.iterrows():
    print(f"\n═══ {r['Model']}  (threshold = {r['Threshold']}) ═══")
    for k in metric_keys:
        print(f"  {k:<25s}: {r[k]}  {r[k + ' 95% CI']}")
    print(f"  {'Training Time':<25s}: {r['Training Time (s)']}s")
    print(f"  {'Inference Time':<25s}: {r['Inference Time (s)']}s")

print(f"\n✓ Saved → {TABLES_DIR / 'table_11_1_rq1_performance.csv'}")

### Calibration Curves (Mục 11.4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, m_name, probs in [
    (axes[0], "Logistic Regression", lr_prob_test),
    (axes[1], "Random Forest",       rf_prob_test),
]:
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)
    ax.plot(prob_pred, prob_true, "s-", label=m_name)
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(f"Calibration Curve — {m_name}")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "calibration_curves.png", dpi=150)
plt.show()
print(f"✓ Saved → {FIGURES_DIR / 'calibration_curves.png'}")

## ✅ RQ1 — HOÀN TẤT!

- Point estimates computed ✓
- Paired Bootstrap CI (1000 resamples) ✓
- Table 11.1 saved ✓
- Calibration curves saved ✓